# Clean, Source-Organized Copy
Saved outputs and machine-specific metadata are intentionally excluded. The byte-identical original remains in the local audit archive. Prepare the local workbench before running this notebook. Notebooks 01 and 02 execute GPU inference when explicitly enabled; 03 disables its optional GPU activation check by default.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts" / "lab.py").is_file()
)
WORKSPACE = PROJECT_ROOT / ".local" / "workbench"
if not (WORKSPACE / "weight_edit.py").is_file():
    raise RuntimeError("Run python scripts/lab.py prepare --with-local-data first.")
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))


# 07 - Published p50c Completion Comparison

Read-only analysis of the fixed edited run and its saved unedited thinking-on control. No model loading, generation, weight edits or downloads occur here. Natural EOS is primary; strict correct delivery is secondary; thinking closure is diagnostic. GSM8K TRAIN is likely in Qwen's training data, so absolute accuracy is not a capability claim.

In [ ]:
import json
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
ROOT = cwd.parent.parent if cwd.name.startswith('p50c_completion_') and cwd.parent.name == 'results' else cwd
if not (ROOT / 'p50c_completion_outputs.py').is_file():
    raise RuntimeError('Open this notebook from the project or its approved run directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p50c_completion_io import RUN
from p50c_completion_outputs import (
    diagnostic_markdown, load_saved_analysis, paired_markdown,
    public_summary, short_wrong_markdown, tables_markdown,
)
from budget_sweep_io import read_json, sha256

RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    raise RuntimeError('This is a read-only notebook; use the explicitly approved guarded worker for execution.')
plan, pool, records, saved, analysis, budget = load_saved_analysis(RUN)
assert 'torch' not in sys.modules and 'transformers' not in sys.modules
print('Loaded verified saved records:', len(records), 'edited + 30 original')
print('No inference performed. Accounted edited tokens:', budget['accounted_tokens'])

## Public Summary And Outcomes

All tables below are recomputed from raw saved token IDs/text, not copied from the report. Wilson intervals are pointwise; they do not apply to mean tokens.

In [ ]:
display(Markdown(public_summary(analysis, saved)))
display(Markdown(tables_markdown(analysis)))

In [ ]:
figure_path = RUN / 'p50c_vs_original_token_budget.png'
finalization = read_json(RUN / 'finalization.json')
assert sha256(figure_path) == finalization['artifact_sha256'][figure_path.relative_to(ROOT).as_posix()]
display(Image(filename=str(figure_path)))

## Paired Effects

Same question ID and seed, edited versus original at each fixed cap. Exact McNemar and Holm adjustment across five caps separately per outcome; secondary/diagnostic evidence cannot replace the primary endpoint.

In [ ]:
display(Markdown(paired_markdown(analysis)))

## Provenance, Closure And Short Wrong Answers

In [ ]:
expected = plan['edit']['target_edited_sha256']
assert all(row['verified_edited_tensor_sha256'] == expected and row['edit_verified_before_generation'] is True for row in records)
assert saved['target_hash_checks']['verified_after_restore'] is True
print('All edited record hashes match:', json.dumps(expected, indent=2))
print('Original target restoration:', saved['target_hash_checks']['verified_after_restore'])
display(Markdown(diagnostic_markdown(analysis)))
display(Markdown(short_wrong_markdown(analysis)))
display(Markdown('### Per-Question Events\n\n' + '\n'.join(
    f"- {row['id']} (seed {row['seed']}): edited EOS={row['edited']['graded']['natural_eos_index']}, "
    f"closure at 8192={row['edited']['caps']['8192']['thinking_closed']}, "
    f"correct delivery={row['edited']['caps']['8192']['strict_correct_delivery']}"
    for row in analysis['per_question']
)))

In [ ]:
from p50c_completion_outputs import amendment_markdown
display(Markdown(amendment_markdown(analysis, saved)))

## Stop And Limitations

Only 30 TRAIN development items, one seed per item, one pinned 0.8B model. The baseline was already observed, the article-based closure estimate remains unverified, and the original and edited runs occurred at different times. The inherited prefix checks are not fresh edited-prefix validation. No held-out use, model saves, downloads, installations, extra conditions or further generation are scheduled.